In [1]:
# 02_make_sector_map.py
import FinanceDataReader as fdr
import pandas as pd
import os

# ==========================================
# ⚙️ 설정
# ==========================================
MARKET = 'US_ALL'  # 'US_ALL' 또는 'KR_ALL'
SAVE_DIR = f"./Raw_Data/{MARKET}"
os.makedirs(SAVE_DIR, exist_ok=True)
# ==========================================

def make_sector_map():
    print(f"📥 [{MARKET}] 섹터 정보 수집 중...")
    
    sector_df = pd.DataFrame()

    if MARKET == 'US_ALL':
        # 1. NASDAQ (Industry, Sector 정보 포함)
        print("   - NASDAQ 정보 수집...")
        nasdaq = fdr.StockListing('NASDAQ')
        
        # 2. S&P500
        print("   - S&P500 정보 수집...")
        sp500 = fdr.StockListing('S&P500')
        
        # 병합
        # 보통 fdr US 결과에는 'Industry'와 'Sector' 컬럼이 있습니다.
        # 없을 경우를 대비해 컬럼 확인
        cols = ['Symbol', 'Name', 'Industry', 'Sector']
        
        df_list = []
        for df in [nasdaq, sp500]:
            # 존재하는 컬럼만 선택
            avail = [c for c in cols if c in df.columns]
            df_list.append(df[avail])
            
        sector_df = pd.concat(df_list)
        sector_df.rename(columns={'Symbol': 'Code'}, inplace=True)

    elif MARKET == 'KR_ALL':
        # KRX 전체 (코스피+코스닥+코넥스)
        print("   - KRX 전체 정보 수집...")
        krx = fdr.StockListing('KRX')
        
        # 한국장은 'Sector'가 '업종' 등의 이름일 수 있음 -> 'Sector'로 통일
        # KRX 리스트 컬럼: Code, Name, Market, Sector, Industry(없을수도)
        if 'Sector' in krx.columns:
            cols = ['Code', 'Name', 'Sector']
        else:
            # 예전 버전 호환용
            cols = ['Code', 'Name']
            
        sector_df = krx[cols]

    # --- 데이터 정리 ---
    # 중복 제거
    sector_df.drop_duplicates(subset=['Code'], inplace=True)
    
    # 결측치 채우기 (ETF 등은 섹터가 없을 수 있음)
    if 'Sector' in sector_df.columns:
        sector_df['Sector'] = sector_df['Sector'].fillna('ETF/Others')
    else:
        sector_df['Sector'] = 'Unknown'
        
    if 'Industry' in sector_df.columns:
        sector_df['Industry'] = sector_df['Industry'].fillna('-')
    else:
        sector_df['Industry'] = '-'

    # 저장
    save_path = f"{SAVE_DIR}/sector_map.csv"
    sector_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    
    print(f"\n✅ 섹터 맵 생성 완료!")
    print(f"📁 저장 위치: {save_path}")
    print(f"📊 총 {len(sector_df)}개 종목 정보 저장됨")

if __name__ == "__main__":
    make_sector_map()

📥 [US_ALL] 섹터 정보 수집 중...
   - NASDAQ 정보 수집...


100%|██████████| 3823/3823 [00:03<00:00, 1064.29it/s]


   - S&P500 정보 수집...

✅ 섹터 맵 생성 완료!
📁 저장 위치: ./Raw_Data/US_ALL/sector_map.csv
📊 총 4166개 종목 정보 저장됨
